In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder
from sklearn.compose import ColumnTransformer

In [2]:
from credit_risk.dataset import AFTER_EDA, load_splits
from credit_risk.features import CATEGORICAL_COLS, prep_one_split, DROP_COLS, NUMERICAL_COLS

2026-06-14 16:52:11.877 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [3]:
from credit_risk.evaluation import evaluate_model

In [4]:
train_df, val_df, test_df, _ = load_splits(path=AFTER_EDA)

2026-06-14 16:52:12.394 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-06-14 16:52:12.401 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-06-14 16:52:12.659 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [5]:
# regime 2 dropping int_rate, grade and sub_grade
CATEGORICAL_COLS.remove('grade')
CATEGORICAL_COLS.remove('sub_grade')
NUMERICAL_COLS.remove('int_rate')

In [10]:
DROP_COLS.remove('zip_code')
DROP_COLS.remove('emp_title')
DROP_COLS.remove('title')
TARGET_ENC = ['zip_code', 'title', 'emp_title']

In [11]:
filtered_train_df, y_train = prep_one_split(df=train_df, drop_cols=DROP_COLS)
filtered_val_df, y_val = prep_one_split(df=val_df, drop_cols=DROP_COLS)
filtered_test_df, y_test = prep_one_split(df=test_df, drop_cols=DROP_COLS)

2026-06-14 16:54:15.061 | INFO     | credit_risk.features:prep_one_split:207 - Inside Function: prep_one_split
2026-06-14 16:54:15.061 | INFO     | credit_risk.features:split_target_and_features:143 - Inside Function: split_target_and_features
2026-06-14 16:54:15.061 | INFO     | credit_risk.features:split_target_and_features:144 - Splitting the target and the features...
2026-06-14 16:54:15.063 | INFO     | credit_risk.features:split_target_and_features:147 - features and target are splitted successfully...
2026-06-14 16:54:15.063 | INFO     | credit_risk.features:add_credit_yrs:160 - Inside Function: add_credit_yrs
2026-06-14 16:54:15.063 | INFO     | credit_risk.features:add_credit_yrs:162 - Adding the feature 'credit_yrs'
2026-06-14 16:54:15.073 | INFO     | credit_risk.features:add_credit_yrs:164 - 'credit_age_yrs added successfully!'
2026-06-14 16:54:15.074 | INFO     | credit_risk.features:add_fico_mid:169 - Inside Function: add_fico_mid
2026-06-14 16:54:15.074 | INFO     | cred

In [12]:
def add_loan_to_income(df: pd.DataFrame) -> pd.DataFrame:
    df['loan_to_income'] = df['loan_amnt'].divide(df['annual_inc'] + 1)
    return df

def add_ins_to_income(df: pd.DataFrame) -> pd.DataFrame:
    df['ins_to_income'] = (df['installment']*12).divide(df['annual_inc'] + 1)
    return df

In [13]:
MISSING_INFO = ['mths_since_last_record', 'mths_since_last_major_derog', 'mths_since_last_major_derog', 'tot_cur_bal']

In [14]:
filtered_train_df = add_loan_to_income(filtered_train_df)
filtered_train_df = add_ins_to_income(filtered_train_df)

filtered_val_df = add_loan_to_income(filtered_val_df)
filtered_val_df = add_ins_to_income(filtered_val_df)

filtered_test_df = add_loan_to_income(filtered_test_df)
filtered_test_df = add_ins_to_income(filtered_test_df)

In [15]:
NUMERICAL_COLS.extend(['loan_to_income', 'ins_to_income'])

In [16]:
num_pipeline = Pipeline([
    ('inpute', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

cat_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

tar_pipeline = Pipeline([
    ('target_enc', TargetEncoder(target_type='binary')),
    ('scalar', StandardScaler())
])

indicator_pipeline = Pipeline([
    ('ind', MissingIndicator(features='all'))
])

In [17]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, NUMERICAL_COLS),
    ('cat', cat_pipeline, CATEGORICAL_COLS),
    ('ind', indicator_pipeline, MISSING_INFO),
    ('tar', tar_pipeline, TARGET_ENC)
])

In [18]:
X_train_feat = preprocessor.fit_transform(X=filtered_train_df, y=y_train.to_numpy())

In [19]:
X_val_feat = preprocessor.transform(filtered_val_df)
X_test_feat = preprocessor.transform(filtered_test_df)

In [20]:
filtered_train_df.shape, filtered_val_df.shape, filtered_test_df.shape

((466042, 73), (420204, 73), (431712, 73))

In [21]:
X_train_feat.shape, X_val_feat.shape, X_test_feat.shape

((466042, 156), (420204, 156), (431712, 156))

In [22]:
from credit_risk.utils import to_jsonable, train

In [23]:
train_proba, val_proba, test_proba = train(
    X_train=X_train_feat,
    y_train=y_train.to_numpy(),
    X_val=X_val_feat,
    X_test=X_test_feat,
    model='LR'
)

In [24]:
lr_eval_dict = evaluate_model(
    y_train=y_train.to_numpy(),
    train_proba=train_proba,
    y_val=y_val.to_numpy(),
    val_proba=val_proba,
    y_test=y_test.to_numpy(),
    test_proba=test_proba,
    fn_cost=10000,
    fp_cost=2000
)

In [25]:
lr_eval_dict

{'threshold': np.float64(0.15000000000000002),
 'train': {'ROC-AUC': 0.700094281580566,
  'PR-AUC': 0.31196057850198267,
  'brier_score': 0.12867557817019185,
  'precision': 0.25228885988257055,
  'recall': 0.7049294413299826,
  'confusion_matrix': array([[226335, 162112],
         [ 22896,  54699]])},
 'val': {'ROC-AUC': 0.7021628623988165,
  'PR-AUC': 0.3398624437572836,
  'brier_score': 0.13806344730912615,
  'precision': 0.2722113245751946,
  'recall': 0.7247644596503505,
  'confusion_matrix': array([[193519, 149522],
         [ 21238,  55925]])},
 'test': {'ROC-AUC': 0.6878734210075416,
  'PR-AUC': 0.299665033687905,
  'brier_score': 0.13179436594868787,
  'precision': 0.2527564385275746,
  'recall': 0.6770378307096349,
  'confusion_matrix': array([[213203, 145711],
         [ 23511,  49287]])}}

In [26]:
train_proba, val_proba, test_proba = train(
    X_train=X_train_feat,
    y_train=y_train.to_numpy(),
    X_val=X_val_feat,
    X_test=X_test_feat,
    model='RF'
)

In [27]:
rf_eval_dict = evaluate_model(
    y_train=y_train.to_numpy(),
    train_proba=train_proba,
    y_val=y_val.to_numpy(),
    val_proba=val_proba,
    y_test=y_test.to_numpy(),
    test_proba=test_proba,
    fn_cost=10000,
    fp_cost=2000
)

In [28]:
rf_eval_dict

{'threshold': np.float64(0.18000000000000002),
 'train': {'ROC-AUC': 1.0,
  'PR-AUC': 1.0,
  'brier_score': 0.018267516876161374,
  'precision': 0.9625738103508162,
  'recall': 1.0,
  'confusion_matrix': array([[385430,   3017],
         [     0,  77595]])},
 'val': {'ROC-AUC': 0.6840443743899787,
  'PR-AUC': 0.31653134987251597,
  'brier_score': 0.1402222280130603,
  'precision': 0.2651594283253219,
  'recall': 0.6941539338802276,
  'confusion_matrix': array([[194601, 148440],
         [ 23600,  53563]])},
 'test': {'ROC-AUC': 0.6658642776826823,
  'PR-AUC': 0.28082889164049263,
  'brier_score': 0.13372833486213032,
  'precision': 0.24107347656548667,
  'recall': 0.6496744416055387,
  'confusion_matrix': array([[210024, 148890],
         [ 25503,  47295]])}}

In [29]:
train_proba, val_proba, test_proba = train(
    X_train=X_train_feat,
    y_train=y_train.to_numpy(),
    X_val=X_val_feat,
    X_test=X_test_feat,
    model='XGB'
)

In [30]:
xgb_eval_dict = evaluate_model(
    y_train=y_train.to_numpy(),
    train_proba=train_proba,
    y_val=y_val.to_numpy(),
    val_proba=val_proba,
    y_test=y_test.to_numpy(),
    test_proba=test_proba,
    fn_cost=10000,
    fp_cost=2000
)

In [31]:
xgb_eval_dict

{'threshold': np.float64(0.15000000000000002),
 'train': {'ROC-AUC': 0.7646020477636788,
  'PR-AUC': 0.4197115182084978,
  'brier_score': 0.1194702684879303,
  'precision': 0.28137209890655596,
  'recall': 0.774018944519621,
  'confusion_matrix': array([[235053, 153394],
         [ 17535,  60060]])},
 'val': {'ROC-AUC': 0.7031086858689606,
  'PR-AUC': 0.3365533083200795,
  'brier_score': 0.1384151130914688,
  'precision': 0.2728793227876495,
  'recall': 0.725231004496974,
  'confusion_matrix': array([[193926, 149115],
         [ 21202,  55961]])},
 'test': {'ROC-AUC': 0.6871411762930006,
  'PR-AUC': 0.2991451854546059,
  'brier_score': 0.13211455941200256,
  'precision': 0.25086555326774956,
  'recall': 0.6778208192532762,
  'confusion_matrix': array([[211563, 147351],
         [ 23454,  49344]])}}

In [32]:
cwd = Path.cwd()
project_root = cwd.parent

In [35]:
lr_path = project_root / 'reports' / 'after_FE'
lr_path.mkdir(parents=True, exist_ok=True)
with open(str(lr_path / 'after_fe_lr.json'), 'w') as f:
    json.dump(to_jsonable(lr_eval_dict), f, indent=2)

In [36]:
with open(str(lr_path / 'after_fe_rf.json'), 'w') as f:
    json.dump(to_jsonable(rf_eval_dict), f, indent=2)

In [37]:
with open(str(lr_path / 'after_fe_xgb.json'), 'w') as f:
    json.dump(to_jsonable(xgb_eval_dict), f, indent=2)